# Data preparation of res ninja data

detailed description of data on https://www.renewables.ninja/

Creates the following parsed datasets

- res_ninja_profiles.csv: hourly capacity factors for pv and on- and offshore wind (1980-2019)

note:
- res ninja has only average capacity factors for some countries. these have been used to fill on- and offshore values

Update 23.07.2025
Data download now only available as zip file from: https://www.renewables.ninja/downloads

In [1]:
#define start and end year of final csv
start_year = 2018
end_year = 2024
defined_years = list(range(start_year, end_year+1))
tyndp_years = [1982,1984,2007]
years = tyndp_years + list(range(start_year, end_year+1))

In [2]:
#Define if we want to download the data
download = "no"

In [3]:
import pandas as pd
import numpy as np
import datetime as dt
import requests
import glob

c:\Users\jonas\anaconda3\Lib\site-packages\pandas\core\arrays\masked.py:61: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


In [4]:
dir_onshore = "../source_data/res_ninja/onshore/"
dir_offshore = "../source_data/res_ninja/offshore/"
dir_pv = "../source_data/res_ninja/pv/"
dir_out = "../parsed_data/"
fn_out = dir_out+"res_ninja_profiles.csv"
fn_out_tyndp = dir_out+"res_ninja_profiles_tyndp_years.csv"

In [5]:
countries = ['AT', 'BE', 'BG', 'HR', 'CZ', 'DK', 'EE', 'FI', 'FR',
             'DE', 'GR', 'HU', 'IE', 'IT', 'LV', 'LT', 'LU', 'NL',
             'PL', 'PT', 'RO', 'SK', 'SI', 'ES', 'SE', 'CH', 'GB', 'NO']

In [6]:
# Download PV data
if download == "yes":
    for country in countries:
        url = "https://www.renewables.ninja/country_downloads/"+country+"/ninja-pv-country-"+country+"-national-merra2.csv"
        response = requests.get(url)
        if response.status_code == 200:
            with open(dir_pv + "ninja-pv-country-"+country+"-national-merra2.csv", "wb") as f:
                f.write(response.content)
        else:
            print(f"File not available for {country}, skipping.")

In [7]:
# Download onshore wind data
if download == "yes":
    for country in countries:
        url = "https://www.renewables.ninja/country_downloads/"+country+"/ninja-wind-country-"+country+"-future_onshore-merra2.csv"
        response = requests.get(url)
        if response.status_code == 200:
            with open(dir_onshore + "ninja-wind-country-"+country+"-future_onshore-merra2.csv", "wb") as f:
                f.write(response.content)
        else:
            print(f"File not available for {country}, skipping.")

In [8]:
# Download offshore wind data
if download == "yes":
    for country in ["BE"]:
        url = "https://www.renewables.ninja/country_downloads/"+country+"/ninja-wind-country-"+country+"-future_offshore-merra2.csv"
        response = requests.get(url)
        if response.status_code == 200:
            with open(dir_offshore + "ninja-wind-country-"+country+"-future_offshore-merra2.csv", "wb") as f:
                f.write(response.content)
        else:
            print(f"File not available for {country}, skipping.")

In [9]:
# create list of files
fns_pv = glob.glob(dir_pv+"*.csv")
fns_onshore = glob.glob(dir_onshore+"*.csv")
fns_offshore = glob.glob(dir_offshore+"*.csv")

In [10]:
# Create dataframe for PV data, we filter the years specified in the years list 
df_pv = pd.DataFrame()
for file in fns_pv:
    df_pv_temp = pd.read_csv(file,parse_dates=True,index_col="time",skiprows=3)
    df_pv_temp['country'] = file[-22:][:2]
    df_pv_temp = df_pv_temp[df_pv_temp.index.year.isin(years)]
    df_pv_temp = df_pv_temp.reset_index().set_index(['time','country'])
    df_pv = pd.concat([df_pv, df_pv_temp])
df_pv = df_pv.rename(columns={'NATIONAL':'Solar'})
df_pv.head(1)

,,Solar
time,country,
1982-01-01 00:00:00+00:00,AT,0.0


In [11]:
# Create dataframe for onshore wind data, we filter the years specified in the years list 
df_onshore = pd.DataFrame()
for file in fns_onshore:
    df_onshore_temp = pd.read_csv(file,parse_dates=True,index_col="time",skiprows=3)
    df_onshore_temp['country'] = file[-28:][:2]
    df_onshore_temp = df_onshore_temp[['country', 'NATIONAL']]
    df_onshore_temp = df_onshore_temp[df_onshore_temp.index.year.isin(years)]
    df_onshore_temp = df_onshore_temp.reset_index().set_index(['time','country'])
    df_onshore = pd.concat([df_onshore, df_onshore_temp])
df_onshore = df_onshore.rename(columns={'NATIONAL':'WindOnshore'})
df_onshore.tail(1)

,,WindOnshore
time,country,
2024-12-31 23:00:00+00:00,SK,0.17629


In [12]:
# Create dataframe for offshore wind data
df_offshore = pd.DataFrame()
for file in fns_offshore:
    df_offshore_temp = pd.read_csv(file,parse_dates=True,index_col="time",skiprows=3)
    df_offshore_temp['country'] = file[-29:][:2]
    df_offshore_temp = df_offshore_temp[['country', 'NATIONAL']]
    df_offshore_temp = df_offshore_temp[df_offshore_temp.index.year.isin(years)]
    df_offshore_temp = df_offshore_temp.reset_index().set_index(['time','country'])
    df_offshore = pd.concat([df_offshore, df_offshore_temp])
df_offshore = df_offshore.rename(columns={'NATIONAL':'WindOffshore'})
df_offshore.tail(1)

,,WindOffshore
time,country,
2024-12-31 23:00:00+00:00,SE,0.607949


In [14]:
#merge all dfs
df_res = pd.merge(df_pv, df_onshore, left_index=True, right_index=True, how='outer').fillna(0)
df_res = pd.merge(df_res, df_offshore, left_index=True, right_index=True, how='outer').fillna(0)
df_res.head()

Solar  WindOnshore  WindOffshore
time                      country                                  
1982-01-01 00:00:00+00:00 AT         0.0     0.056055       0.00000
                          BE         0.0     0.126363       0.12109
                          BG         0.0     0.161991       0.00000
                          CH         0.0     0.080239       0.00000
                          CZ         0.0     0.255405       0.00000

In [15]:
#we limit years for smaller file size
df_res_out = df_res[df_res.index.get_level_values('time').year.isin(defined_years)]
df_res_out.head()

Solar  WindOnshore  WindOffshore
time                      country                                  
2018-01-01 00:00:00+00:00 AT         0.0     0.249404      0.000000
                          BE         0.0     0.870866      0.949667
                          BG         0.0     0.248403      0.000000
                          CH         0.0     0.497442      0.000000
                          CZ         0.0     0.429303      0.000000

In [16]:
#we also create a dataset including only the three TYNDP weather years
df_res_out_tyndp = df_res[df_res.index.get_level_values('time').year.isin(tyndp_years)]
df_res_out_tyndp.head()

Solar  WindOnshore  WindOffshore
time                      country                                  
1982-01-01 00:00:00+00:00 AT         0.0     0.056055       0.00000
                          BE         0.0     0.126363       0.12109
                          BG         0.0     0.161991       0.00000
                          CH         0.0     0.080239       0.00000
                          CZ         0.0     0.255405       0.00000

In [17]:
#finally, transform from wide to long format
df_res_out_long = pd.melt(df_res_out.reset_index(),id_vars=['time','country'],var_name='tech', value_name='capacity_factor')
df_res_out_long.head()

,time,country,tech,capacity_factor
0,2018-01-01 00:00:00+00:00,AT,Solar,0.0
1,2018-01-01 00:00:00+00:00,BE,Solar,0.0
2,2018-01-01 00:00:00+00:00,BG,Solar,0.0
3,2018-01-01 00:00:00+00:00,CH,Solar,0.0
4,2018-01-01 00:00:00+00:00,CZ,Solar,0.0


In [18]:
df_res_out_long_tyndp = pd.melt(df_res_out_tyndp.reset_index(),id_vars=['time','country'],var_name='tech', value_name='capacity_factor')
df_res_out_long_tyndp.head()

,time,country,tech,capacity_factor
0,1982-01-01 00:00:00+00:00,AT,Solar,0.0
1,1982-01-01 00:00:00+00:00,BE,Solar,0.0
2,1982-01-01 00:00:00+00:00,BG,Solar,0.0
3,1982-01-01 00:00:00+00:00,CH,Solar,0.0
4,1982-01-01 00:00:00+00:00,CZ,Solar,0.0


In [19]:
# Create CSV file
df_res_out_long.to_csv(fn_out,index=False)

In [20]:
df_res_out_long_tyndp.to_csv(fn_out_tyndp,index=False) 